In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder

In [2]:
SEED = 42
np.random.seed(SEED)

In [3]:
train_df = pd.read_csv("../data/student_query_train_5000.csv").drop(columns=["ID"])
test_df = pd.read_csv("../data/student_query_test_1000.csv").drop(columns=["ID"])

In [4]:
le = LabelEncoder()
train_df["y"] = le.fit_transform(train_df["Priority"])
test_df["y"] = le.transform(test_df["Priority"])
print("Sınıf eşlemesi:", dict(zip(le.classes_, range(len(le.classes_)))))

Sınıf eşlemesi: {'High': 0, 'Low': 1, 'Medium': 2}


In [5]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tr_idx, val_idx = next(gss.split(train_df, groups=train_df["Student_Query"]))

In [13]:
tr_df = train_df.iloc[tr_idx].reset_index(drop=True)
val_df = train_df.iloc[val_idx].reset_index(drop=True)

In [14]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [18]:
tokenizer = Tokenizer(oov_token="<OOV>", lower=True)
tokenizer.fit_on_texts(tr_df["Student_Query"])

In [19]:
vocab_size = len(tokenizer.word_index) + 1
print("Vocab boyutu (padding dahil):", vocab_size)

Vocab boyutu (padding dahil): 101


In [20]:
tr_seq = tokenizer.texts_to_sequences(tr_df["Student_Query"])
val_seq = tokenizer.texts_to_sequences(val_df["Student_Query"])
test_seq = tokenizer.texts_to_sequences(test_df["Student_Query"])

In [21]:
MAXLEN = max(len(s) for s in tr_seq)
print("MAXLEN:", MAXLEN)

MAXLEN: 12


In [22]:
X_tr = pad_sequences(tr_seq, maxlen=MAXLEN, padding="pre")
X_val = pad_sequences(val_seq, maxlen=MAXLEN, padding="pre")
X_test = pad_sequences(test_seq, maxlen=MAXLEN, padding="pre")

In [23]:
def oov_rate(texts):
    total, oov = 0, 0
    oov_idx = tokenizer.word_index["<OOV>"]
    for s in tokenizer.texts_to_sequences(texts):
        total += len(s)
        oov += sum(1 for t in s if t == oov_idx)
    return oov / total if total else 0.0

In [24]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import classification_report, confusion_matrix

In [25]:
tf.keras.utils.set_random_seed(SEED)


In [26]:
NUM_CLASSES = len(le.classes_)


In [27]:
model = models.Sequential([
    layers.Input(shape=(MAXLEN,)),
    layers.Embedding(input_dim=vocab_size, output_dim=64, mask_zero=True),
    layers.LSTM(64, dropout=0.2),          # girdi bağlantılarında dropout
    layers.Dropout(0.3),                   # yoğun katman öncesi dropout
    layers.Dense(NUM_CLASSES, activation="softmax"),
])

In [28]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0),
    loss="sparse_categorical_crossentropy",   # y tam sayı olduğu için sparse
    metrics=["accuracy"],
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 12, 64)              │           6,464 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ (None, 64)                  │          33,024 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 3)                   │             195 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 39,683 (155.01 KB)

 Trainable params: 39,683 (155.01 KB)

 Non-trainable params: 0 (0.00 B)

In [29]:
early = callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)


In [30]:
history = model.fit(
    X_tr, tr_df["y"].values,
    validation_data=(X_val, val_df["y"].values),
    epochs=30,
    batch_size=64,
    callbacks=[early],
    verbose=2,
)

Epoch 1/30
62/62 - 13s - 208ms/step - accuracy: 0.9104 - loss: 0.5048 - val_accuracy: 0.3486 - val_loss: 1.3925
Epoch 2/30
62/62 - 4s - 62ms/step - accuracy: 1.0000 - loss: 0.0034 - val_accuracy: 0.5124 - val_loss: 1.8705
Epoch 3/30
62/62 - 2s - 40ms/step - accuracy: 1.0000 - loss: 0.0012 - val_accuracy: 0.5124 - val_loss: 2.0460
Epoch 4/30
62/62 - 2s - 35ms/step - accuracy: 1.0000 - loss: 7.3808e-04 - val_accuracy: 0.5124 - val_loss: 2.2303


In [31]:
def evaluate(name, X, y):
    probs = model.predict(X, verbose=0)
    pred = probs.argmax(axis=1)
    print(f"\n===== {name} =====")
    print("Accuracy:", round((pred == y).mean(), 4))
    print(classification_report(y, pred, target_names=le.classes_, digits=3))
    print("Confusion matrix (satır=gerçek, sütun=tahmin):")
    print(confusion_matrix(y, pred))
    return pred

_ = evaluate("Doğrulama", X_val, val_df["y"].values)
_ = evaluate("Test", X_test, test_df["y"].values)


===== Doğrulama =====
Accuracy: 0.3486
              precision    recall  f1-score   support

        High      1.000     0.258     0.411       712
         Low      0.538     1.000     0.700       182
      Medium      0.000     0.000     0.000       156

    accuracy                          0.349      1050
   macro avg      0.513     0.419     0.370      1050
weighted avg      0.771     0.349     0.400      1050

Confusion matrix (satır=gerçek, sütun=tahmin):
[[184   0 528]
 [  0 182   0]
 [  0 156   0]]

===== Test =====
Accuracy: 0.863
              precision    recall  f1-score   support

        High      1.000     0.687     0.815       326
         Low      0.906     1.000     0.951       337
      Medium      0.748     0.896     0.815       337

    accuracy                          0.863      1000
   macro avg      0.884     0.861     0.860      1000
weighted avg      0.883     0.863     0.861      1000

Confusion matrix (satır=gerçek, sütun=tahmin):
[[224   0 102]
 [  0 337

In [32]:
model.save("../model/ques_prio.keras")